# Tokenization

Tokenization is a 'necessary evil', on one hand it's the reason for many problems with LLMs (why old models werent' good at coding, why spelling is hard for LLMs, why changes of language affect an llm's capabilities...), but on the other side, since the Time dimension in the transformer architecture is computationally constrained, we want to have our tokens as large as possible to get most of our computations. The goal of tokenization is to find the sweet spot / compremize allowing us to have a good vocabulary while keeping a good quality of the llm answers.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [19]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text)
print("length:", len(text))
print('---')
print(tokens)
print("length:", len(tokens))

---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

## Byte pair encoding algo:

In [20]:
# we want to apply the byte pair encoding algo

idx = 256
translator = {}
inversetrans = {}
tokenlist = []

def maxoccur(tokens):
    d = {}
    for pair in zip(tokens, tokens[1:]):   
        d[pair] = d.get(pair,0) + 1

    token = max(d, key = d.get)
    return token

# tochange = maxoccur(tokens)
# print(tochange)

# idx = idx + 1

# translator[idx] = tochange

#we then replace the token with a new one the duplicates:
def new_tokens(tokens, tochange):
    i = 0
    new_tokens = []
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == tochange:
            new_tokens.append(idx)
            i = i + 2
        else :
            new_tokens.append(tokens[i])
            i = i+1

    return new_tokens

# print(new_tokens(tokens, tochange))



In [21]:
# now we train the byte pair algo:
while len(tokens)  > 400:
    tochange = maxoccur(tokens)
    

    idx = idx + 1

    translator[tochange] = idx
    inversetrans[idx] = tochange
    tokenlist.append(tochange)
    print("added " , tochange, "as character" , idx )
    tokens = new_tokens(tokens, tochange)

print(tokenlist)

added  (101, 32) as character 257
added  (240, 159) as character 258
added  (226, 128) as character 259
added  (105, 110) as character 260
added  (115, 32) as character 261
added  (97, 110) as character 262
added  (116, 104) as character 263
added  (258, 133) as character 264
added  (258, 135) as character 265
added  (97, 114) as character 266
added  (239, 189) as character 267
added  (259, 140) as character 268
added  (268, 265) as character 269
added  (101, 114) as character 270
added  (111, 114) as character 271
added  (116, 32) as character 272
added  (260, 103) as character 273
added  (115, 116) as character 274
added  (262, 100) as character 275
added  (32, 263) as character 276
added  (44, 32) as character 277
added  (97, 109) as character 278
added  (276, 257) as character 279
added  (111, 117) as character 280
added  (85, 110) as character 281
added  (281, 105) as character 282
added  (282, 99) as character 283
added  (283, 111) as character 284
added  (284, 100) as character 

In [29]:
# now suppose we want to tokenize text:

text1 = "This algo alone runs into issues, you might end up with a vocab containing many versions of the same word : `dog.`, `dog!`, `dog?` etc... Which is inefficient and wasteful.One way to counter this, is to force the tokenizer to unconcat them, using regex for example."

######################## WRONG CODE; KEEPING IT BECAUSE IT CONTAINS A USEFUL MISTAKE I MADE
######################## HERE I USED A GREEDY ALGO TO PICK TOKENS TO JOIN
######################## HOWEVER, JOINING TOKENS SHOULD BE DONE BASED ON PRIORITY!

#def tokenizer(text):
#     tokens = text.encode("utf-8") # raw bytes
#     tokens = list(map(int, tokens))
#     i = 0
#     ans = []
    
#     while i < len(tokens) -1: # abcdefg 
#         if (tokens[i] , tokens[i+1]) in translator:
            
#             val1 = translator[(tokens[i] , tokens[i+1])]
#             i = i+1
#             while i < len(tokens) -1 and (val1 , tokens[i+1]) in translator:
#                 val1 = translator[(val1 , tokens[i+1])]
#                 i = i+1
#             ans.append(val1)
#             i = i + 1
#         else :
#             ans.append(tokens[i])
#             i = i + 1
#     if i == len(tokens) -1 :
#         ans.append(tokens[i])
        

#     return ans
# 

# print(len(text1), len(tokenizer(text1)))


    
    


In [35]:
def tokenizer(text):
    tokens = text.encode("utf-8") # raw bytes
    tokens = list(map(int, tokens))
   
    
    for duo in tokenlist:
        copy = []
        i = 0
        while i < len(tokens) - 1:
            if (tokens[i], tokens[i+1]) == duo :
                
                copy.append(translator[duo])
                i = i + 2
            else:
                copy.append(tokens[i])
                i = i+1
        if i == len(tokens) -1 :
            copy.append(tokens[i])
        
        tokens = copy.copy()
    return tokens

text2 = tokenizer(text1)

print(text2)

print(len(text1.encode("utf-8")),len(tokenizer(text1)) )
    
        

    

[84, 104, 105, 261, 97, 108, 103, 111, 32, 97, 108, 111, 110, 257, 114, 117, 110, 261, 290, 111, 32, 105, 115, 115, 117, 101, 286, 121, 280, 32, 109, 105, 103, 104, 272, 101, 110, 100, 32, 117, 112, 32, 119, 105, 263, 32, 97, 32, 118, 111, 99, 97, 98, 32, 99, 111, 110, 116, 97, 260, 273, 32, 109, 262, 121, 32, 118, 270, 115, 105, 111, 110, 261, 111, 102, 279, 115, 278, 257, 119, 271, 100, 32, 58, 32, 96, 100, 111, 103, 46, 96, 277, 96, 100, 111, 103, 33, 96, 277, 96, 100, 111, 103, 63, 96, 32, 101, 116, 99, 46, 46, 46, 32, 87, 104, 105, 99, 104, 32, 105, 261, 260, 101, 102, 102, 105, 99, 105, 101, 110, 272, 289, 119, 97, 274, 101, 102, 117, 108, 46, 79, 110, 257, 119, 97, 121, 32, 116, 111, 32, 99, 280, 110, 116, 270, 276, 105, 286, 105, 261, 116, 111, 32, 102, 271, 99, 257, 263, 257, 116, 111, 107, 101, 110, 105, 122, 270, 32, 116, 111, 32, 117, 110, 99, 111, 110, 99, 97, 272, 263, 101, 109, 277, 117, 115, 273, 32, 114, 101, 103, 101, 120, 32, 102, 271, 32, 101, 120, 278, 112, 108, 10

In [ ]:
# now the detokenizer:

def detokenizer(tokens):
    
    def toroots(token):
        
        
        if token in inversetrans:
            return toroots(inversetrans[token][0]) + toroots(inversetrans[token][1])
        else :
            return [token]

    
    ans = []
    for token in tokens:
        ans =  ans + toroots(token)
    let = [bytes([elem]) for elem in ans]
    let = b"".join(let)
    return let.decode('utf-8', errors="replace")


    
print(detokenizer(text2))

#as you can see, our detokenizer and our tokenizer both work!
#small note : the code is ineficient for tokenizer (should've kept a priority list for pairs). 
# coded this fast before watching karpathy's sol and wanted to keep the code that way (same for why i kept the greedy algo).
 

print(text1 == detokenizer(tokenizer(text1)))

    

This algo alone runs into issues, you might end up with a vocab containing many versions of the same word : `dog.`, `dog!`, `dog?` etc... Which is inefficient and wasteful.One way to counter this, is to force the tokenizer to unconcat them, using regex for example.
True


This algo alone runs into issues, you might end up with a vocab containing many versions of the same word : `dog.`, `dog!`, `dog?` etc... Which is inefficient and wasteful.
One way to counter this, is to force the tokenizer to unconcat them, using regex for example.

## Sentencepiece

Diff vs tiktoken: you can use it for both training and tokenizing / detokenizing. Was developed way before the LLM era, so has some weird parameters like the sentences params. Still quite used in the industry (Mistral and LLama for example). Karpathy doesn't like it x)). 


## More about tokenization

- Why bad at spelling / gpt 2 bad in python / bad at arithmetics...
- Adding customized tokens in finetuning
- Compressing prompts with custom tokens
- Tokenization with images/video
- Out of sample prompts (breaking the model)